# Vue d'Ensemble : Analyse de Robustesse Adversariale

Ce notebook charge les modèles pré-entraînés (Baseline, Transformer, Robuste)
et démontre leur performance sur des données clean et leur vulnérabilité 
(ou résistance) aux attaques adversariales.

In [ ]:
import pandas as pd
import joblib
import sys
from pathlib import Path

In [ ]:
# (Assumer que le notebook est lancé depuis le dossier 'notebooks/')
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
# Importer les wrappers de modèle (nécessite de les copier depuis les scripts ou de les mettre dans src/)
# Pour cet exemple, nous allons les redéfinir ici pour la simplicité.
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from textattack.models.wrappers import ModelWrapper, SklearnModelWrapper, HuggingFaceModelWrapper

class TFKerasModelWrapper(HuggingFaceModelWrapper):
    """ Wrapper TextAttack pour les modèles Keras TFAutoModel... """
    def __call__(self, text_input_list):
        inputs = self.tokenizer(
            text_input_list, 
            return_tensors="tf", 
            padding=True, 
            truncation=True
        )
        logits = self.model(inputs["input_ids"])[0]
        return logits.numpy()

## 1. Chargement des Données et Modèles

In [ ]:
# Charger les données de test
test_df = pd.read_parquet("../data/processed/test.parquet")
print(f"{len(test_df)} exemples de test chargés.")

In [ ]:
# Définir quelques exemples pour les tests
sample_texts = [
    "This movie was absolutely fantastic, the acting was superb!", # Positif
    "I hated this film. It was boring and the plot was predictable." # Négatif
]
sample_labels = [1, 0]

In [ ]:
# Charger le modèle Baseline (Sklearn)
model_baseline = joblib.load("../artifacts/models/baseline_logreg.joblib")
print("Modèle Baseline (LogReg) chargé.")

In [ ]:
# Charger le modèle Robuste (Sklearn)
model_robust = joblib.load("../artifacts/models/baseline_logreg_robust.joblib")
print("Modèle Robuste (LogReg) chargé.")

In [ ]:
# Charger le modèle Transformer
model_path = "../artifacts/models/distilbert_baseline"
tokenizer_path = "../artifacts/tokenizers/distilbert_baseline"
model_tf = TFAutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer_tf = AutoTokenizer.from_pretrained(tokenizer_path)
print("Modèle Transformer (DistilBERT) chargé.")

## 2. Prédictions (Données Clean)

In [ ]:
def predict_sklearn(model, texts):
    preds = model.predict(texts)
    probs = model.predict_proba(texts)
    return ["Positif" if p == 1 else "Négatif" for p in preds], probs

def predict_transformer(model, tokenizer, texts):
    inputs = tokenizer(texts, return_tensors="tf", padding=True, truncation=True)
    logits = model(inputs).logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    preds = tf.argmax(logits, axis=-1).numpy()
    return ["Positif" if p == 1 else "Négatif" for p in preds], probs

In [ ]:
print("--- Prédictions Baseline ---")
preds_base, probs_base = predict_sklearn(model_baseline, sample_texts)
for text, pred, prob in zip(sample_texts, preds_base, probs_base):
    print(f"  {pred} ({max(prob)*100:.1f}%) -> '{text[:50]}...'")

In [ ]:
print("\n--- Prédictions Transformer ---")
preds_tf, probs_tf = predict_transformer(model_tf, tokenizer_tf, sample_texts)
for text, pred, prob in zip(sample_texts, preds_tf, probs_tf):
    print(f"  {pred} ({max(prob)*100:.1f}%) -> '{text[:50]}...'")

## 3. Attaque (Live Demo)

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019
from textattack.attacker import Attacker
from textattack.attack_args import AttackArgs

In [ ]:
# Wrapper le modèle baseline
wrapper_baseline = SklearnModelWrapper(model_baseline)

In [ ]:
# Wrapper le modèle robuste
wrapper_robust = SklearnModelWrapper(model_robust)

In [ ]:
# Configurer l'attaque
attack_recipe = TextFoolerJin2019.build(wrapper_baseline)
dataset = [(sample_texts[0], sample_labels[0])] # Attaquer le premier exemple positif

attack_args = AttackArgs(num_examples=1, disable_stdout=True)

In [ ]:
# Attaquer le modèle BASELINE
print("Attaque sur le modèle Baseline (LogReg)...")
attacker_baseline = Attacker(attack_recipe, dataset, attack_args)
result_baseline = attacker_baseline.attack_dataset()[0]

print("\n--- Résultat Attaque Baseline ---")
print(result_baseline.summary(color_method='ansi'))

In [ ]:
# Attaquer le modèle ROBUSTE
# Note: L'attaque doit cibler le modèle qu'elle attaque
attack_recipe_robust = TextFoolerJin2019.build(wrapper_robust)

print("\nAttaque sur le modèle Robuste (LogReg)...")
attacker_robust = Attacker(attack_recipe_robust, dataset, attack_args)
result_robust = attacker_robust.attack_dataset()[0]

print("\n--- Résultat Attaque Robuste ---")
print(result_robust.summary(color_method='ansi'))

**Analyse :**
* Le modèle **Baseline** a probablement été trompé (Original -> Flipped).
* Le modèle **Robuste** a de meilleures chances d'avoir résisté (Original -> Skipped/Failed).

## 4. Chargement des Résultats d'Évaluation

In [ ]:
import json

try:
    with open("../artifacts/reports/robustness_summary.json", 'r') as f:
        results = json.load(f)
    
    # Transformer en DataFrame Pandas pour un bel affichage
    rows = []
    for model, attacks in results.items():
        for attack, metrics in attacks.items():
            row = {
                "Modèle": model,
                "Attaque": attack,
                "Acc (Clean)": metrics["Accuracy (Clean)"] * 100,
                "Acc (Advers.)": metrics["Accuracy (Adversarial)"] * 100,
                "ASR (%)": metrics["Attack Success Rate"] * 100
            }
            rows.append(row)
            
    results_df = pd.DataFrame(rows)
    display(results_df.pivot(index="Modèle", columns="Attaque", values=["Acc (Advers.)", "ASR (%)"]))
    
except FileNotFoundError:
    print("Le rapport 'robustness_summary.json' n'a pas été trouvé.")
    print("Veuillez exécuter : python scripts/6_evaluate_robustness.py")